# 01 Data Acquisition

Executable reporting notebook for the Google Play data acquisition phase. It reads existing local summaries, raw-count evidence, aggregate EDA metrics, and saved figures. It does not scrape Google Play and it handles ignored raw files gracefully.


## CRISP-DM Stage

Data Understanding. This notebook validates the Google Play review acquisition output before preprocessing.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "CLAUDE.md").exists() and (candidate / "ml-service").exists():
            return candidate
    raise RuntimeError("Could not find SentiRank project root from current working directory.")


PROJECT_ROOT = find_project_root()
ML_SERVICE_DIR = PROJECT_ROOT / "ml-service"
DATASETS_DIR = PROJECT_ROOT / "datasets"
DOCS_FIGURES_DIR = PROJECT_ROOT / "docs" / "figures"


def load_json(path: Path):
    if not path.exists():
        display(Markdown(f"Missing JSON: `{path.relative_to(PROJECT_ROOT)}`"))
        return None
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def load_csv(path: Path, **kwargs):
    if not path.exists():
        display(Markdown(f"Missing CSV: `{path.relative_to(PROJECT_ROOT)}`"))
        return None
    return pd.read_csv(path, **kwargs)


def show_json_as_table(payload, title: str):
    if payload is None:
        return
    display(Markdown(f"### {title}"))
    try:
        table = pd.json_normalize(payload, sep=".").T.reset_index()
        table.columns = ["field", "value"]
        display(table)
    except Exception as exc:
        display(Markdown(f"Could not tabulate JSON payload: `{exc}`"))
        display(payload)


def show_csv_preview(path: Path, title: str, rows: int = 10):
    data = load_csv(path)
    if data is None:
        return None
    display(Markdown(f"### {title}"))
    display(data.head(rows))
    if len(data) > rows:
        display(Markdown(f"Rows: `{len(data)}`. Showing first `{rows}` rows."))
    return data


def show_figures(figure_paths: list[Path], title: str):
    display(Markdown(f"## {title}"))
    shown = 0
    for figure_path in figure_paths:
        if figure_path.exists():
            display(Markdown(f"**{figure_path.relative_to(PROJECT_ROOT)}**"))
            display(Image(filename=str(figure_path)))
            shown += 1
        else:
            display(Markdown(f"Missing figure: `{figure_path.relative_to(PROJECT_ROOT)}`"))
    if shown == 0:
        display(Markdown("No figures were available to display."))

print(f"Project root: {PROJECT_ROOT}")


## Optional Safe Script Check

The only acquisition script command included here is a dry run. Keep `RUN_SCRAPE_DRY_RUN = False` unless you specifically want to inspect the planned scraper configuration. This cell never runs live scraping.


In [ ]:
import subprocess
import sys

RUN_SCRAPE_DRY_RUN = False

if RUN_SCRAPE_DRY_RUN:
    command = [sys.executable, "scripts/scrape_reviews.py", "--dry-run"]
    result = subprocess.run(command, cwd=ML_SERVICE_DIR, text=True, capture_output=True, check=False)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f"Return code: {result.returncode}")
else:
    display(Markdown("Dry-run command is available but skipped by default: `python scripts/scrape_reviews.py --dry-run`."))


## Load Acquisition Summaries

These files are generated artifacts and may be absent on a fresh clone because raw outputs are ignored by Git.


In [ ]:
RAW_DIR = DATASETS_DIR / "raw"
summary_files = {
    "Data acquisition summary": RAW_DIR / "data_acquisition_summary.json",
    "Scraping summary": RAW_DIR / "scraping_summary.json",
}

acquisition_summaries = {name: load_json(path) for name, path in summary_files.items()}
for name, payload in acquisition_summaries.items():
    show_json_as_table(payload, name)


## Raw Per-Rating Row Counts

Counts are computed only if `reviews_rating_1_raw.csv` through `reviews_rating_5_raw.csv` exist locally.


In [ ]:
TARGET_QUOTA = {1: 20000, 2: 15000, 3: 30000, 4: 15000, 5: 20000}
row_count_records = []

for rating, target in TARGET_QUOTA.items():
    csv_path = RAW_DIR / f"reviews_rating_{rating}_raw.csv"
    if csv_path.exists():
        count = len(pd.read_csv(csv_path))
        status = "available"
    else:
        count = None
        status = "missing ignored local file"
    row_count_records.append({
        "rating": rating,
        "target_rows": target,
        "actual_rows": count,
        "achievement_pct": (count / target * 100) if count is not None and target else None,
        "status": status,
    })

raw_counts_df = pd.DataFrame(row_count_records)
display(raw_counts_df)

available_total = raw_counts_df["actual_rows"].dropna().sum()
if available_total:
    display(Markdown(f"Available local raw rows: `{int(available_total):,}`."))


## EDA Metrics

Aggregate metrics are read from `datasets/outputs/eda/01_data_acquisition/`. These files are small dashboard/thesis artifacts.


In [ ]:
EDA01_DIR = DATASETS_DIR / "outputs" / "eda" / "01_data_acquisition"
metric_files = [
    "rating_distribution_raw.csv",
    "sentiment_distribution_raw.csv",
    "text_length_histogram_raw.csv",
    "temporal_distribution_monthly_raw.csv",
    "temporal_distribution_monthly_by_rating.csv",
    "scraping_quota_achievement.csv",
    "missing_value_summary.csv",
]

loaded_eda01_tables = {}
for filename in metric_files:
    table = show_csv_preview(EDA01_DIR / filename, filename, rows=12)
    if table is not None:
        loaded_eda01_tables[filename] = table

for filename in ["text_length_summary_raw.json", "rating_distribution_raw.json", "sentiment_distribution_raw.json", "scraping_quota_achievement.json", "missing_value_summary.json"]:
    show_json_as_table(load_json(EDA01_DIR / filename), filename)


## EDA Figures

Figures are loaded from `docs/figures/01_data_acquisition/`.


In [ ]:
FIG01_DIR = DOCS_FIGURES_DIR / "01_data_acquisition"
figure_files = [
    "rating_distribution_raw.png",
    "sentiment_distribution_raw.png",
    "text_length_histogram_raw.png",
    "text_length_boxplot_raw.png",
    "temporal_distribution_raw.png",
    "temporal_distribution_by_rating_raw.png",
    "scraping_quota_achievement.png",
    "missing_value_summary.png",
]
show_figures([FIG01_DIR / filename for filename in figure_files], "Data Acquisition Figures")


## Interpretation Notes

The cell below derives thesis-facing interpretation from the available summaries and row counts.


In [ ]:
notes = []

summary = acquisition_summaries.get("Data acquisition summary") or {}
rows_per_rating = summary.get("rows_per_rating") or {}
if rows_per_rating:
    notes.append(f"Summary rows per rating: `{rows_per_rating}`.")
elif raw_counts_df["actual_rows"].notna().any():
    rows_from_files = {int(row.rating): int(row.actual_rows) for row in raw_counts_df.itertuples() if pd.notna(row.actual_rows)}
    notes.append(f"Rows per rating from local raw CSV files: `{rows_from_files}`.")
else:
    notes.append("Raw per-rating CSV files are not available in this checkout; this is expected when ignored raw data is not copied locally.")

rating_3_row = raw_counts_df.loc[raw_counts_df["rating"] == 3].iloc[0]
if pd.notna(rating_3_row["actual_rows"]):
    rating_3_actual = int(rating_3_row["actual_rows"])
    rating_3_target = int(rating_3_row["target_rows"])
    if rating_3_actual < rating_3_target:
        notes.append(f"Rating 3 reached `{rating_3_actual:,}` of `{rating_3_target:,}` target rows, so Neutral availability is a documented data limitation.")
    else:
        notes.append("Rating 3 reached the configured target quota.")
else:
    quota_table = loaded_eda01_tables.get("scraping_quota_achievement.csv")
    if quota_table is not None and "rating" in quota_table.columns:
        rating_3_quota = quota_table.loc[quota_table["rating"] == 3]
        if not rating_3_quota.empty:
            notes.append(f"Rating 3 quota evidence from metrics: `{rating_3_quota.to_dict(orient='records')[0]}`.")

for key in ["duplicate_external_id_count", "missing_content_count", "missing_rating_count"]:
    if key in summary:
        notes.append(f"{key}: `{summary[key]}`.")

notes.append("Next step: continue with `02_preprocessing.ipynb` for relabeling, text normalization, and preprocessing EDA.")

display(Markdown("\n".join(f"- {note}" for note in notes)))
